In [1]:
import pandas as pd
import urllib.parse
import seaborn as sns
import matplotlib.pyplot as plt
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Carrega as variáveis do arquivo oculto .env
load_dotenv()

# Puxando as credenciais com segurança
db_user = os.getenv("DB_USER")
db_password = urllib.parse.quote_plus(os.getenv("DB_PASSWORD"))
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# Monta a URI corrigida (Corrigido: db_port)
POSTGRES_URI = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(POSTGRES_URI)

In [5]:
# Puxando os dados do Materialized view
df_ml = pd.read_sql_query('SELECT * FROM mv_saude_tech_integrada ORDER BY "userId", data_referencia', engine)

In [6]:
# Engenharia de Recursos
# Criando uma vriável alvo: 0 estresse de amanhã foi alto (4 ou 5)
df_ml['estresse_alto_amanha'] = df_ml.groupby("userId")['nivel_estresse'].shift(-1).apply(
    lambda x: 1 if x >= 4 else 0 if pd.notna(x) else None
)

# Removendo as linhas que ficaram sem o "dia seguinte"
df_ml_clean = df_ml.dropna(subset=['avg_rhr', 'sleep_score', 'minutes_asleep', 'estresse_alto_amanha']).copy()

# Separação de atributos (X) e alvo (y)
X = df_ml_clean[['avg_rhr', 'sleep_score', 'minutes_asleep']]
y = df_ml_clean['estresse_alto_amanha'].astype(int)

# Divisão de Treino e Teste (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

print(f"Dados de Treino: {X_train.shape[0]} dias | Dados de Teste: {X_test.shape[0]} dias")

Dados de Treino: 434370 dias | Dados de Teste: 108593 dias


In [11]:
# Treinando o Modelo (Árvore de decisão)
modelo_burnout = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_burnout.fit(X_train, y_train)

# Avaliação da IA
predicao = modelo_burnout.predict(X_test)

print("Relatório de performance do modelo:")
print()
print(classification_report(y_test, predicao, target_names=['Risco Baixo', 'Risco de Estresse Alto']))


Relatório de performance do modelo:

                        precision    recall  f1-score   support

           Risco Baixo       0.83      1.00      0.91     89468
Risco de Estresse Alto       0.93      0.05      0.09     19125

              accuracy                           0.83    108593
             macro avg       0.88      0.52      0.50    108593
          weighted avg       0.85      0.83      0.76    108593



In [12]:
# Importância das variáveis (o que o modelo achou mais importante)
importancias = pd.DataFrame({'Atributo': X.columns,
                             'Importancia': modelo_burnout.feature_importances_}).sort_values(by='Importancia', ascending=False)

print(" O que a IA considerou mais crítico para prever o estresse?")
display(importancias)

 O que a IA considerou mais crítico para prever o estresse?


,Atributo,Importancia
0,avg_rhr,0.548065
2,minutes_asleep,0.329226
1,sleep_score,0.122708
